# 1.1 ETL Pipeline
Extracts raw SimFin data, computes technical features, adds the target variable, and saves one processed CSV per ticker to `../data/processed/`.

In [90]:
import os
import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────
# Edit INPUT_PATH to where you put the SimFin bulk download files
INPUT_PATH = "../data/raw/us-shareprices-daily.csv"
OUTPUT_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load raw share prices

In [91]:
df_share_prices = pd.read_csv(INPUT_PATH, delimiter=";")
print("Shape:", df_share_prices.shape)
df_share_prices.head()

Shape: (6205521, 11)


,Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding
0,A,45846,2020-04-07,76.61,77.56,74.02,74.03,71.17,2458009,NaN,309651359.0
1,A,45846,2020-04-08,74.17,77.17,72.75,76.69,73.73,2702954,NaN,309651359.0
2,A,45846,2020-04-09,76.43,78.72,76.23,78.33,75.31,2399863,NaN,309651359.0
3,A,45846,2020-04-13,77.44,77.99,75.02,76.21,73.27,1533000,NaN,309651359.0
4,A,45846,2020-04-14,77.30,79.20,77.24,78.83,75.79,2650262,NaN,309651359.0


## Load companies list

In [92]:
df_companies = pd.read_csv("../data/raw/us-companies.csv", delimiter=";")
print("Shape:", df_companies.shape)
# Sort by number of employees to get a sense of the biggest companies
df_companies.sort_values("Number Employees", ascending=False).head(10)

Shape: (6531, 11)


,Ticker,SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
399,AMZN,62747,AMAZON COM INC,103002.0,US0231351067,12.0,1298000.0,Amazon.com Inc is an online retailer. The Comp...,us,1018724.0,USD
6008,UPS,106423,UNITED PARCEL SERVICE INC,100010.0,US9113121068,12.0,543000.0,United Parcel Service Inc is a package deliver...,us,1090727.0,USD
120,ACN,61372,Accenture plc,108002.0,IE00B4BNMY34,8.0,506000.0,Accenture PLC is a professional service compan...,us,1467373.0,USD
2827,HOME,233408,At Home Group Inc.,103002.0,US04650Y1001,1.0,400000.0,At Home Group Inc is a home décor superstore. ...,us,1646228.0,USD
5692,TESS,42813,TESSCO TECHNOLOGIES INC,101005.0,US8723861071,3.0,391500.0,Tessco Technologies Inc architects and deliver...,us,927355.0,USD
961,BRK-A,71306,BERKSHIRE HATHAWAY INC,111001.0,US0846701086,12.0,371653.0,"Berkshire Hathaway Inc., through its subsidiar...",us,1067983.0,USD
2308,FMX,18773575,"Fomento Económico Mexicano, S.A.B. de C.V.",102004.0,US3444191064,12.0,350000.0,"Fomento Económico Mexicano, S.A.B. de C.V., th...",us,1061736.0,USD
5129,SBUX,86689,STARBUCKS CORP,103003.0,US8552441094,9.0,349000.0,"Starbucks Corp is the roaster, marketer and re...",us,829224.0,USD
2944,IBM,69543,INTERNATIONAL BUSINESS MACHINES CORP,101003.0,US4592001014,12.0,345900.0,International Business Machines Corp offers a ...,us,51143.0,USD
5794,TNET,797860,TRINET GROUP INC,100002.0,US8962881079,12.0,334600.0,Trinet Group Inc provides human resources solu...,us,937098.0,USD


## Step 1 — Filter to one company
Develop and test on AMZN first, then scale to all tickers.

In [93]:
def filter_company(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Filter to one ticker and sort chronologically."""
    company_df = df[df["Ticker"] == ticker].copy()
    company_df.sort_values("Date", ascending=True, inplace=True)
    company_df.reset_index(drop=True, inplace=True)
    return company_df

df_amzn = filter_company(df_share_prices, "AMZN")
print(f"AMZN rows: {len(df_amzn)}")
df_amzn.head()

AMZN rows: 1238


,Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding
0,AMZN,62747,2020-04-07,100.86,101.79,99.88,100.58,100.58,102279660,NaN,9.980000e+09
1,AMZN,62747,2020-04-08,101.05,102.20,100.56,102.15,102.15,79546260,NaN,9.980000e+09
2,AMZN,62747,2020-04-09,102.22,102.65,100.88,102.14,102.14,93112340,NaN,9.980000e+09
3,AMZN,62747,2020-04-13,102.00,109.00,101.90,108.44,108.44,134334180,NaN,9.980000e+09
4,AMZN,62747,2020-04-14,110.02,114.60,109.31,114.17,114.17,161743860,NaN,9.980000e+09


## Step 2 — Add technical features

In [94]:
def add_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute technical features."""
    df = df.copy()
    close = df["Close"]
    high = df["High"]
    low = df["Low"]
    volume = df["Volume"]
    
    # ── YOUR EXISTING FEATURES ──
    df["Returns"] = np.log(close / close.shift(1))
    df["SMA_5"] = close.rolling(5).mean()
    df["SMA_20"] = close.rolling(20).mean()
    df["Volatility_5"] = df["Returns"].rolling(5).std()
    df["Volatility_20"] = df["Returns"].rolling(20).std()
    df["Volume_Change"] = volume.pct_change()
    
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    rs = gain.rolling(14).mean() / loss.rolling(14).mean()
    df["RSI_14"] = 100 - (100 / (1 + rs))
    df["Price_Range"] = (high - low) / close
    
    # ── NEW FEATURES TO ADD ──
    
    # 1. MACD (Moving Average Convergence Divergence)
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema_12 - ema_26
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]
    
    # 2. Bollinger Bands
    sma_20 = close.rolling(20).mean()
    std_20 = close.rolling(20).std()
    df["BB_Upper"] = sma_20 + (2 * std_20)
    df["BB_Lower"] = sma_20 - (2 * std_20)
    df["BB_Width"] = (df["BB_Upper"] - df["BB_Lower"]) / sma_20
    df["BB_Position"] = (close - df["BB_Lower"]) / (df["BB_Upper"] - df["BB_Lower"])
    
    # 3. Momentum indicators
    df["Momentum_10"] = close / close.shift(10) - 1
    df["Momentum_20"] = close / close.shift(20) - 1
    
    # 4. Average True Range (ATR) - measures volatility
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df["ATR_14"] = tr.rolling(14).mean()
    df["ATR_Ratio"] = df["ATR_14"] / close  # Normalized ATR
    
    # 5. Lagged returns (what happened in previous days)
    df["Return_Lag1"] = df["Returns"].shift(1)
    df["Return_Lag2"] = df["Returns"].shift(2)
    df["Return_Lag3"] = df["Returns"].shift(3)
    df["Return_Lag5"] = df["Returns"].shift(5)
    
    # 6. Volume features
    df["Volume_SMA_10"] = volume.rolling(10).mean()
    df["Volume_Ratio"] = volume / df["Volume_SMA_10"]
    
    # 7. Day of week (markets behave differently on different days)
    df["Date"] = pd.to_datetime(df["Date"])
    df["DayOfWeek"] = df["Date"].dt.dayofweek
    
    # 8. Distance from moving averages
    df["Dist_SMA_5"] = (close - df["SMA_5"]) / df["SMA_5"]
    df["Dist_SMA_20"] = (close - df["SMA_20"]) / df["SMA_20"]
    
    return df


In [95]:
FEATURE_COLS = [
    # Original
    "Returns", "SMA_5", "SMA_20", "Volatility_5", "Volatility_20",
    "Volume_Change", "RSI_14", "Price_Range",
    # New
    "MACD", "MACD_Signal", "MACD_Hist",
    "BB_Width", "BB_Position",
    "Momentum_10", "Momentum_20",
    "ATR_Ratio",
    "Return_Lag1", "Return_Lag2", "Return_Lag3", "Return_Lag5",
    "Volume_Ratio",
    "DayOfWeek",
    "Dist_SMA_5", "Dist_SMA_20",
]

df_amzn = add_technical_features(df_amzn)
print(f"Shape after features: {df_amzn.shape}")
df_amzn[FEATURE_COLS].head()

Shape after features: (1238, 39)


,Returns,SMA_5,SMA_20,Volatility_5,Volatility_20,Volume_Change,RSI_14,Price_Range,MACD,MACD_Signal,...,Momentum_20,ATR_Ratio,Return_Lag1,Return_Lag2,Return_Lag3,Return_Lag5,Volume_Ratio,DayOfWeek,Dist_SMA_5,Dist_SMA_20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.018990,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
1,0.015489,NaN,NaN,NaN,NaN,-0.222267,NaN,0.016055,0.125242,0.025048,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN
2,-0.000098,NaN,NaN,NaN,NaN,0.170543,NaN,0.017329,0.221141,0.064267,...,NaN,NaN,0.015489,NaN,NaN,NaN,NaN,3,NaN,NaN
3,0.059853,NaN,NaN,NaN,NaN,0.442711,NaN,0.065474,0.796320,0.210678,...,NaN,NaN,-0.000098,0.015489,NaN,NaN,NaN,0,NaN,NaN
4,0.051492,105.496,NaN,NaN,NaN,0.204041,NaN,0.046334,1.694978,0.507538,...,NaN,NaN,0.059853,-0.000098,0.015489,NaN,NaN,1,0.082221,NaN


## Step 3 — Add target variable
`Target = 1` if tomorrow's Close > today's Close, else `0`.
This is the binary label the ML model will learn to predict.

In [96]:
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create TWO target columns:
    
    Target_Binary (2 classes):
        0 = Fall (return <= 0)
        1 = Rise (return > 0)
    
    Target_Multi (4 classes):
        0 = Big Fall   (return < -1%)
        1 = Small Fall (return between -1% and 0%)
        2 = Small Rise (return between 0% and +1%)
        3 = Big Rise   (return > +1%)
    """
    df = df.copy()
    
    # Calculate next-day return
    next_day_return = (df["Close"].shift(-1) / df["Close"]) - 1
    
    # ── Binary Target ──
    df["Target_Binary"] = (next_day_return > 0).astype(float)
    
    # ── Multi-Class Target ──
    def classify_multi(ret):
        if pd.isna(ret):
            return np.nan
        elif ret < -0.01:
            return 0  # Big Fall
        elif ret < 0:
            return 1  # Small Fall
        elif ret < 0.01:
            return 2  # Small Rise
        else:
            return 3  # Big Rise
    
    df["Target_Multi"] = next_day_return.apply(classify_multi)
    
    # Drop rows where we can't calculate target (last row)
    df = df.dropna(subset=["Target_Binary", "Target_Multi"])
    df["Target_Binary"] = df["Target_Binary"].astype(int)
    df["Target_Multi"] = df["Target_Multi"].astype(int)
    
    return df

## Step 4 — Drop columns with many NaN values
Removes columns where more than 75% of values are missing (e.g. Dividend).

In [97]:
def drop_columns_many_nans(df: pd.DataFrame, threshold: float = 0.75) -> pd.DataFrame:
    """Drop columns where the fraction of NaN values exceeds the threshold."""
    return df.loc[:, df.isna().mean() < threshold]

df_amzn = drop_columns_many_nans(df_amzn, threshold=0.75)
print("Remaining columns:", df_amzn.columns.tolist())
print("Any remaining NaN:", df_amzn.isnull().sum().sum())

Remaining columns: ['Ticker', 'SimFinId', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj. Close', 'Volume', 'Shares Outstanding', 'Returns', 'SMA_5', 'SMA_20', 'Volatility_5', 'Volatility_20', 'Volume_Change', 'RSI_14', 'Price_Range', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Position', 'Momentum_10', 'Momentum_20', 'ATR_14', 'ATR_Ratio', 'Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Return_Lag5', 'Volume_SMA_10', 'Volume_Ratio', 'DayOfWeek', 'Dist_SMA_5', 'Dist_SMA_20']
Any remaining NaN: 252


## Step 5 — Save processed file

In [98]:
def save_processed(df: pd.DataFrame, output_dir: str, ticker: str) -> str:
    """Save the processed DataFrame to CSV. Returns the file path."""
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"{ticker.lower()}_processed.csv")
    df.to_csv(path, index=False)
    print(f"[ETL] Saved {ticker}: {len(df)} rows → {path}")
    return path

save_processed(df_amzn, OUTPUT_DIR, "AMZN")

[ETL] Saved AMZN: 1238 rows → ../data/processed/amzn_processed.csv


'../data/processed/amzn_processed.csv'

## Step 6 — Full pipeline function
Orchestrates all steps above into one call.

In [99]:
def extract_share_prices(input_path: str) -> pd.DataFrame:
    """Load the raw SimFin share-prices CSV."""
    return pd.read_csv(input_path, delimiter=";")


def run_etl(input_path: str, output_dir: str, ticker: str) -> pd.DataFrame:
    """
    Execute the full ETL pipeline for a single ticker.

    Parameters
    ----------
    input_path : path to us-shareprices-daily.csv
    output_dir : where to save the processed CSV
    ticker     : e.g. 'AMZN'

    Returns
    -------
    Processed DataFrame (also saved to disk).
    """
    raw = extract_share_prices(input_path)
    df  = filter_company(raw, ticker)

    if df.empty:
        raise ValueError(f"No data found for ticker '{ticker}'.")

    df = add_technical_features(df)
    df = add_target(df)
    df = drop_columns_many_nans(df, threshold=0.75)

    save_processed(df, output_dir, ticker)
    return df

## Step 7 — Run ETL for all 5 tickers
Change this list to match whichever companies your group chose.

In [100]:
TICKERS = ["AMZN", "AAPL", "MSFT", "GOOG", "TSLA"]

for ticker in TICKERS:
    run_etl(INPUT_PATH, OUTPUT_DIR, ticker)

[ETL] Saved AMZN: 1237 rows → ../data/processed/amzn_processed.csv
[ETL] Saved AAPL: 1237 rows → ../data/processed/aapl_processed.csv
[ETL] Saved MSFT: 1237 rows → ../data/processed/msft_processed.csv
[ETL] Saved GOOG: 1237 rows → ../data/processed/goog_processed.csv
[ETL] Saved TSLA: 1237 rows → ../data/processed/tsla_processed.csv


## Quick sanity check — inspect one processed file

In [101]:
# Verify both targets
for ticker in TICKERS:
    df = pd.read_csv(f"{OUTPUT_DIR}/{ticker.lower()}_processed.csv")
    print(f"\n{ticker}:")
    print(f"  Binary: 0={df['Target_Binary'].eq(0).sum()}, 1={df['Target_Binary'].eq(1).sum()}")
    print(f"  Multi:  0={df['Target_Multi'].eq(0).sum()}, 1={df['Target_Multi'].eq(1).sum()}, "
          f"2={df['Target_Multi'].eq(2).sum()}, 3={df['Target_Multi'].eq(3).sum()}")


AMZN:
  Binary: 0=595, 1=642
  Multi:  0=332, 1=259, 2=266, 3=380

AAPL:
  Binary: 0=577, 1=660
  Multi:  0=266, 1=307, 2=325, 3=339

MSFT:
  Binary: 0=587, 1=650
  Multi:  0=267, 1=318, 2=320, 3=332

GOOG:
  Binary: 0=558, 1=679
  Multi:  0=292, 1=266, 2=341, 3=338

TSLA:
  Binary: 0=588, 1=649
  Multi:  0=435, 1=152, 2=174, 3=476
